In [ ]:
!pip install --upgrade --index-url https://pypi.ngc.nvidia.com nvidia-tensorrt
!sudo apt-get install -y tensorrt

In [ ]:
import tensorrt
print(tensorrt.__version__)
assert tensorrt.Builder(tensorrt.Logger())

In [ ]:
!sudo apt-get install python3-libnvinfer-dev -y
!python3 -m pip install numpy onnx
!sudo apt-get install onnx-graphsurgeon -y
#--------------------------------------------------
#upgrade pytorch to 1.12
#!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

In [ ]:
!dpkg-query -W tensorrt
!dpkg -l | grep nvinfer

In [ ]:
!pip install --upgrade pip
!pip install --upgrade sphinx-glpi-theme
!pip install --no-cache-dir --index-url https://pypi.nvidia.com pytorch-quantization
!pip install wget
!pip3 install -U pip && pip3 install onnxsim
!pip install onnxruntime

In [ ]:
# !pip install git+https://github.com/huggingface/accelerate

In [ ]:
# !pip show accelerate

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
torch.cuda.empty_cache()


In [ ]:
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms as transforms
from torchvision import models, datasets
from torch.utils.tensorboard import SummaryWriter
import onnxruntime
import onnxsim


import pytorch_quantization
from pytorch_quantization import nn as quant_nn
from pytorch_quantization import quant_modules
from pytorch_quantization import calib
from pytorch_quantization.tensor_quant import QuantDescriptor
from pytorch_quantization.nn.modules.tensor_quantizer import TensorQuantizer
from pytorch_quantization.nn.modules import quant_pooling
from tqdm import tqdm

print(pytorch_quantization.__version__)
# print(torch_tensorrt.__version__)
print(onnxruntime.__version__)
print(onnxsim.__version__)
#print(torchaudio.__version__)

import os
import sys
import warnings
import time
import numpy as np
import wget
import tarfile
import shutil
warnings.simplefilter('ignore')
import collections

# Accelerate parts
from accelerate import Accelerator, notebook_launcher # main interface, distributed launcher
from accelerate.utils import set_seed # reproducability across devices

In [ ]:
# Initialize the Accelerator
accelerator = Accelerator()

# Set seed for reproducibility
set_seed(42)


In [ ]:
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os

class CityscapesDataset(Dataset):
    def __init__(self, root_dir, split='train', resize_shape=(256, 512), is_train=True):
        super().__init__()
        self.inputs, self.targets = [], []
        self.resize_shape = resize_shape
        self.ignore_index = 255
        self.valid_classes = [7, 8, 11, 12, 13, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 31, 32, 33]
        self.class_map = {k: v for v, k in enumerate(self.valid_classes)}
        self.is_train = is_train

        img_dir = os.path.join(root_dir, 'leftImg8bit', split)
        mask_dir = os.path.join(root_dir, 'gtFine', split)

        for root, _, filenames in os.walk(img_dir):
            for filename in filenames:
                if filename.endswith('.png'):
                    filename_base = '_'.join(filename.split('_')[:-1])
                    input_path = os.path.join(root, filename_base + '_leftImg8bit.png')
                    target_path = os.path.join(mask_dir, os.path.basename(root), filename_base + '_gtFine_labelIds.png')

                    if os.path.exists(input_path) and os.path.exists(target_path):
                        self.inputs.append(input_path)
                        self.targets.append(target_path)

        print(f"Found {len(self.inputs)} images and {len(self.targets)} targets for split '{split}'")

        # Define Albumentations transformation pipeline
        if self.is_train:
            self.transform = A.Compose([
                A.Resize(self.resize_shape[0], self.resize_shape[1]),  # Ensure consistent size
                A.HorizontalFlip(p=0.5),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
                A.GaussianBlur(p=0.2),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2()
            ])
        else:
            self.transform = A.Compose([
                A.Resize(self.resize_shape[0], self.resize_shape[1]),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2()
            ])

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        input_img = np.array(Image.open(self.inputs[i]).convert('RGB'))
        target_img = np.array(Image.open(self.targets[i]))

        # Apply transformations to both image and mask
        transformed = self.transform(image=input_img, mask=target_img)
        input_img = transformed['image']
        target_tensor = transformed['mask']

        # Remap target labels
        remapped_target = torch.full_like(target_tensor, self.ignore_index)
        for cls, mapped in self.class_map.items():
            remapped_target[target_tensor == cls] = mapped

        # Debugging output for shapes
        assert input_img.shape[1:] == remapped_target.shape, f"Shape mismatch: input {input_img.shape[1:]}, target {remapped_target.shape}"

        return input_img, remapped_target


# DataLoader setup
root_dir = '/kaggle/input/cityscapes/Cityscape'
resize_shape = (512, 1024)

train_dataset = CityscapesDataset(root_dir=root_dir, split='train', resize_shape=resize_shape, is_train=True)
val_dataset = CityscapesDataset(root_dir=root_dir, split='val', resize_shape=resize_shape, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)
inference_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)



In [ ]:
import matplotlib.pyplot as plt
import torch

# Decode function to convert mask to RGB
def decode_segmap(mask, n_classes=20, label_colours=None):
    if label_colours is None:
        label_colours = [
            (102, 102, 156), (70, 70, 70), (244, 35, 232), (128, 64, 128),
            (190, 153, 153), (153, 153, 153), (250, 170, 30), (220, 220, 0),
            (107, 142, 35), (152, 251, 152), (0, 130, 180), (220, 20, 60),
            (255, 0, 0), (0, 0, 142), (0, 0, 70), (0, 60, 100),
            (0, 80, 100), (0, 0, 230), (119, 11, 32), (120, 170, 250)
        ]

    # Convert to numpy array if needed
    if isinstance(mask, torch.Tensor):
        mask = mask.cpu().numpy()

    # Initialize empty RGB channels
    r, g, b = np.zeros_like(mask, dtype=np.uint8), np.zeros_like(mask, dtype=np.uint8), np.zeros_like(mask, dtype=np.uint8)

    # Apply color mapping
    for l in range(n_classes):
        r[mask == l] = label_colours[l][0]
        g[mask == l] = label_colours[l][1]
        b[mask == l] = label_colours[l][2]

    # Stack to form an RGB image
    rgb = np.stack([r, g, b], axis=2)  # Result shape: [height, width, 3]
    return rgb

# Visualization function for input, ground truth, and predicted segmentation
def visualize_segmentation(input_image, decoded_target, decoded_prediction):
    # Display the images
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    axs[0].imshow(input_image.permute(1, 2, 0).cpu().numpy())  # Convert input tensor to numpy for display
    axs[0].set_title("Input Image")
    axs[0].axis('off')

    axs[1].imshow(decoded_target)
    axs[1].set_title("Ground Truth Mask")
    axs[1].axis('off')

    axs[2].imshow(decoded_prediction)
    axs[2].set_title("Predicted Mask")
    axs[2].axis('off')

    plt.show()

# Function to perform inference and visualize predictions
def visualize_predictions(model, data_loader, device, n_classes=20):
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():
        for input_img, target_mask in data_loader:
            # Move data to the appropriate device
            input_img, target_mask = input_img.to(device), target_mask.to(device)

            # Perform inference
            output = model(input_img)['out']  # Assuming DeepLabV3 output format
            _, predicted_mask = torch.max(output, 1)  # Get predicted class indices

            # Convert single image from batch
            input_image = input_img[0]  # Select the first image from the batch
            target_mask = target_mask[0]  # Select the corresponding ground truth mask
            predicted_mask = predicted_mask[0]  # Select the corresponding predicted mask

            # Decode ground truth and prediction masks only once
            decoded_target = decode_segmap(target_mask, n_classes=n_classes)
            decoded_prediction = decode_segmap(predicted_mask, n_classes=n_classes)

            # Visualize the segmentation results
            visualize_segmentation(input_image, decoded_target, decoded_prediction)
            break  # Visualize only one batch for now

# Usage example:
# Assuming `trained_model` is the pre-trained DeepLabV3 model and `device` is the device ('cpu' or 'cuda')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = model.to(device)  # Ensure model is on the correct device

# Call the visualization function with the model and data loader
visualize_predictions(model, inference_loader, device, n_classes=19)


In [ ]:
from torchvision.models.segmentation import lraspp_mobilenet_v3_large

# Load the DeepLabV3-ResNet50 model pre-trained on COCO
# if accelerator.is_main_process:

model = lraspp_mobilenet_v3_large(pretrained=True) # Load the model pre-trained on COCO
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 19  # Cityscapes has 19 valid classes
# Modify low_classifier and high_classifier for 19 classes
# Assuming `model` is your LRASPP-based model
replace_relu_with_silu(model)
model.classifier.low_classifier = torch.nn.Conv2d(40, 19, kernel_size=1, stride=1)
model.classifier.high_classifier = torch.nn.Conv2d(128, 19, kernel_size=1, stride=1)

# Wrap the model in DataParallel and move to main device
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    # model = nn.DataParallel(model)  # Wrap model in DataParallel

# Move the model to the GPU if available
# model = model.to(accelerator.device)
model = model.to(device)


In [ ]:
import torch.nn as nn

def replace_relu_with_silu(module):
    for name, child in module.named_children():
        if isinstance(child, nn.ReLU):
            setattr(module, name, nn.SiLU())
        else:
            replace_relu_with_silu(child)


In [ ]:
import torch
from collections import OrderedDict  # Import OrderedDict


# Load the full checkpoint dictionary
checkpoint = torch.load("/kaggle/input/lsarpp_mobilenetv3_segmentation/pytorch/default/1/baseline_mobilenetv3.pth")

# Remove the "module." prefix from the keys in state_dict if necessary
state_dict = checkpoint['model_state_dict']
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k.replace("module.", "")  # Remove `module.` prefix
    new_state_dict[name] = v

# Load the model state
model.load_state_dict(new_state_dict)

# Load the optimizer state (optional, only if you want to continue training)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Retrieve additional information
epoch = checkpoint['epoch']
val_accuracy = checkpoint['val_accuracy']

print(f"Model loaded from epoch {epoch} with validation accuracy {val_accuracy* 100:.2f}%")

# Move model to the appropriate device if needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set model to evaluation mode if you're using it for inference
model.eval()


In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.lr_scheduler import OneCycleLR

# Loss function with ignore_index for unlabeled pixels
criterion = nn.CrossEntropyLoss(ignore_index=255)
# optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

# Learning rate scheduler (optional, adjust patience based on training observations)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3, verbose=True)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)



optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
# scheduler = OneCycleLR(optimizer, max_lr=1e-3, steps_per_epoch=len(train_loader), epochs=40)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, threshold=0.001, verbose=True)


# Wrap with Accelerator's prepare
# model, optimizer, train_loader, val_loader = accelerator.prepare(
#     model, optimizer, train_loader, val_loader
# )

In [ ]:
from sklearn.metrics import confusion_matrix
from torch.nn import CrossEntropyLoss

# Helper functions for accuracy and IoU
def calculate_accuracy(outputs, targets):
    _, preds = torch.max(outputs, 1)
    correct = (preds == targets).float().sum()
    return correct / targets.numel()


def calculate_mean_iou(preds, targets, n_classes):
    preds = preds.cpu().numpy().flatten()
    targets = targets.cpu().numpy().flatten()
    mask = targets != 255  # Ignore unlabeled pixels
    preds = preds[mask]
    targets = targets[mask]
    if len(preds) == 0:
        return 0.0
    cm = confusion_matrix(targets, preds, labels=np.arange(n_classes))
    intersection = np.diag(cm)
    union = cm.sum(axis=1) + cm.sum(axis=0) - intersection
    iou = intersection / (union + 1e-10)
    return np.nanmean(iou)


# Training loop
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_accuracy = 0.0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device).long()
        optimizer.zero_grad()
        outputs = model(images)  # LRASPP has only 'out'
        main_output = outputs['out']
        loss = criterion(main_output, masks)
        loss.backward()
        optimizer.step()
        accuracy = calculate_accuracy(main_output, masks)
        running_loss += loss.item()
        running_accuracy += accuracy.item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = running_accuracy / len(train_loader)
    return epoch_loss, epoch_accuracy


# Validation loop with mean IoU calculation
def evaluate_model_with_metrics(model, val_loader, criterion, device, n_classes):
    model.eval()
    val_loss, val_accuracy, val_iou = 0.0, 0.0, 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device).long()
            outputs = model(images)
            main_output = outputs['out']
            loss = criterion(main_output, masks)
            accuracy = calculate_accuracy(main_output, masks)
            _, preds = torch.max(main_output, 1)
            iou = calculate_mean_iou(preds, masks, n_classes)
            val_loss += loss.item()
            val_accuracy += accuracy.item()
            val_iou += iou
            torch.cuda.empty_cache()
    avg_val_loss = val_loss / len(val_loader)
    avg_val_accuracy = val_accuracy / len(val_loader)
    avg_iou = val_iou / len(val_loader)
    return avg_val_loss, avg_val_accuracy, avg_iou


# EarlyStopping class for training monitoring
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_accuracy):
        if self.best_score is None:
            self.best_score = val_accuracy
        elif val_accuracy < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_accuracy
            self.counter = 0


# Function to save the model
def save_checkpoint(model, optimizer, epoch, val_accuracy, path='best_model.pth', accelerator=None):
    model_to_save = accelerator.unwrap_model(model) if accelerator is not None else model
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model_to_save.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_accuracy': val_accuracy
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved to {path}")

In [ ]:
import os

# n_classes = 19
num_epochs = 40
patience = 8  # Stop training if no improvement for 5 epochs
early_stopping = EarlyStopping(patience=patience, min_delta=0.001)
best_val_accuracy = 0
log_path = '/kaggle/working/training_log-mobilenetv3.txt'

# Open log file
with open(log_path, 'w') as log_file:
    log_file.write("Epoch, Train Loss, Train Acc, Val Loss, Val Acc, Val IoU\n")

    # Training loop with early stopping
    for epoch in range(num_epochs):
        # Train the model for one epoch
        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, accelerator.device)
        val_loss, val_accuracy, val_iou = evaluate_model_with_metrics(model, val_loader, criterion, accelerator.device, num_classes)

        # Print metrics for the current epoch
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}")
        print(f"Validation Loss: {val_loss:.4f}, Validation Acc: {val_accuracy:.4f}, Validation IoU: {val_iou:.4f}")

        # Log metrics to file
        log_file.write(f"{epoch + 1}, {train_loss:.4f}, {train_accuracy:.4f}, {val_loss:.4f}, {val_accuracy:.4f}, {val_iou:.4f}\n")

        # Save the model checkpoint if validation accuracy improves
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            save_checkpoint(model, optimizer, epoch, val_accuracy, path='/kaggle/working/baseline_mobilenetv3.pth', accelerator=accelerator)

        # Early stopping
        early_stopping(val_accuracy)
        if early_stopping.early_stop:
            print("Early stopping triggered. Stopping training.")
            break

        # Step the scheduler if using a plateau scheduler
        scheduler.step(val_iou)

print("Training complete!")


In [ ]:
val_loss, val_accuracy, val_iou = evaluate_model_with_metrics(model, val_loader, criterion, device, num_classes)
print(f"Validation Loss: {val_loss:.4f}, Validation Acc: {val_accuracy:.4f}, Validation IoU: {val_iou:.4f}")

In [ ]:
val_loss, val_accuracy, val_iou = evaluate_model_with_metrics(model, inference_loader, criterion, device, num_classes)
print(f"Validation Loss: {val_loss:.4f}, Validation Acc: {val_accuracy:.4f}, Validation IoU: {val_iou:.4f}")

In [ ]:
# Check if the model is wrapped in DataParallel
if isinstance(model, torch.nn.DataParallel):
    model = model.module  # Access the original model inside DataParallel


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Function to compute layer sensitivities and parameter sizes
def compute_layer_sensitivity_and_size(model, loss_fn, data_loader, epsilon=1e-3, num_samples=10, device='cuda'):
    sensitivities = {}
    parameter_sizes = {}
    model.to(device).eval()

    for name, layer in model.named_modules():
        if isinstance(layer, nn.Conv2d):
            # Calculate parameter size in MB
            num_params = sum(p.numel() for p in layer.parameters())
            parameter_sizes[name] = num_params * 8 / (8 * 1024 * 1024)  # Convert to MB

            # Compute sensitivity
            total_gradient_norm = 0.0
            original_weights = [p.data.clone() for p in layer.parameters()]

            for batch_idx, (inputs, targets) in enumerate(data_loader):
                inputs = inputs.to(device).float()
                targets = targets.to(device).long()

                for _ in range(num_samples):
                    for p in layer.parameters():
                        perturbation = torch.randn_like(p, device=device) * epsilon / torch.norm(p)
                        p.data += perturbation

                    outputs = model(inputs)
                    main_output = outputs['out'] if isinstance(outputs, dict) else outputs
                    loss = loss_fn(main_output, targets)

                    model.zero_grad()
                    loss.backward()

                    gradient_norm = sum(torch.norm(p.grad, p=1).item() for p in layer.parameters())
                    total_gradient_norm += gradient_norm

                    for p, orig in zip(layer.parameters(), original_weights):
                        p.data.copy_(orig)

                if batch_idx == 0:
                    break

            sensitivities[name] = total_gradient_norm / (num_samples * (batch_idx + 1))

    return sensitivities, parameter_sizes

import matplotlib.pyplot as plt

def dp_optimize(sensitivities, parameter_sizes, bit_precisions, target_model_size):
    """
    Optimize layer bit precisions using dynamic programming and visualize results.

    Args:
        sensitivities (dict): Sensitivity scores for each layer.
        parameter_sizes (dict): Parameter sizes (in MB) for each layer.
        bit_precisions (list): List of available bit precision options.
        target_model_size (float): Target total model size (in MB).

    Returns:
        tuple: Best Pareto-optimal solution and a list of all Pareto-optimal solutions.
    """
    layers = list(sensitivities.keys())
    n_layers = len(layers)
    max_size = int(target_model_size * 1024)  # Convert MB to KB for DP indexing

    # Initialize DP table
    dp_table = [{} for _ in range(n_layers + 1)]
    dp_table[0][0] = (0, {})  # (accuracy_loss, bit_precision_config)

    # Fill DP table
    for i, layer in enumerate(layers, start=1):
        sensitivity = sensitivities[layer]
        param_size = parameter_sizes[layer] * 1024  # Convert MB to KB for DP indexing

        for prev_size, (prev_loss, prev_config) in dp_table[i - 1].items():
            for precision in bit_precisions:
                layer_size = param_size * (precision / 8.0)  # Compute size for the given precision
                new_size = prev_size + int(layer_size)

                if new_size > max_size:
                    continue

                accuracy_loss = prev_loss + sensitivity * (1 / precision)  # Quantization error inversely proportional to precision
                new_config = prev_config.copy()
                new_config[layer] = precision

                if new_size not in dp_table[i] or dp_table[i][new_size][0] > accuracy_loss:
                    dp_table[i][new_size] = (accuracy_loss, new_config)

    # Extract Pareto-optimal solutions
    pareto_solutions = []
    for model_size, (accuracy_loss, bit_config) in dp_table[n_layers].items():
        pareto_solutions.append((model_size / 1024, accuracy_loss, bit_config))  # Convert size back to MB

    pareto_solutions.sort(key=lambda x: x[1])  # Sort by accuracy loss

    # Visualization of Pareto-Frontier
    sizes = [solution[0] for solution in pareto_solutions]
    losses = [solution[1] for solution in pareto_solutions]

    plt.figure(figsize=(10, 6))
    plt.plot(sizes, losses, marker='o', label='Pareto-Frontier')
    plt.xlabel("Model Size (MB)")
    plt.ylabel("Accuracy Loss")
    plt.title("Pareto-Frontier: Accuracy Loss vs. Model Size")
    plt.grid(alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Return the best solution (smallest accuracy loss) and all Pareto solutions
    best_solution = pareto_solutions[0]
    return best_solution, pareto_solutions




In [ ]:

bit_precisions = [8, 4, 2]
target_model_size = 4.0  # Target model size in MB


# sensitivities, parameter_sizes = compute_layer_sensitivity_and_size(model=model, loss_fn =nn.CrossEntropyLoss(ignore_index=255) , 
#                                    data_loader=train_loader, epsilon=1e-3, num_samples=10, device='cuda')
# Optimize and visualize
best_solution, pareto_solutions = dp_optimize(sensitivities, parameter_sizes, bit_precisions, target_model_size)

# Print the best solution
model_size, accuracy_loss, bit_config = best_solution
print(f"Best Solution:\nModel Size: {model_size:.2f} MB, Accuracy Loss: {accuracy_loss:.2f}, Bit Precisions: {bit_config}")

# Print all Pareto-optimal solutions
print("\nPareto-Optimal Solutions:")
for size, loss, config in pareto_solutions:
    print(f"Model Size: {size:.2f} MB, Accuracy Loss: {loss:.2f}, Bit Precisions: {config}")


In [ ]:
import matplotlib.pyplot as plt


def visualize_sensitivity_scores(sensitivities, figsize=(20, 8)):
    """
    Visualize sensitivity scores for model layers.

    Args:
        sensitivity_scores (dict): A dictionary where keys are layer names and values are sensitivity scores.
        log_scale (bool): Whether to plot sensitivity scores in log10 scale. Default is True.
        figsize (tuple): Figure size for the plot. Default is (20, 8).

    Returns:
        None
    """
    # Extract layer names and sensitivity scores
    layers = list(sensitivities.keys())
    scores = list(sensitivities.values())
    
     # Create the plot
    plt.figure(figsize=figsize)
    plt.plot(scores, marker='o', linestyle='-', color='b', label='Sensitivity Scores')
    plt.yscale('log')  # Use logarithmic scale for the Y-axis
    plt.xticks(ticks=range(len(layers)), labels=layers, rotation=90, fontsize=8)
    plt.ylabel("Sensitivity Score (Log Scale)")
    plt.xlabel("Layers")
    plt.title("Sensitivity Scores by Layer (Logarithmic Scale)")
    plt.grid(alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()


# Visualize sensitivity scores
visualize_sensitivity_scores(sensitivities)

In [ ]:
def assign_bit_precision(sensitivity_scores, bit_precisions=(8, 4, 2), thresholds=(0.8, 0.2)):
    """
    Assign bit precision to layers based on sensitivity scores.

    Args:
        sensitivity_scores (dict): Layer sensitivity scores as {layer_name: sensitivity_score}.
        bit_precisions (tuple): Bit precision levels (e.g., 8, 4, 2).
        thresholds (tuple): Normalized thresholds for assigning bit precision (high, low).

    Returns:
        dict: Mapping of layer names to assigned bit precision.
    """
    # Normalize sensitivity scores
    max_score = max(sensitivity_scores.values())
    min_score = min(sensitivity_scores.values())
    normalized_scores = {
        layer: (score - min_score) / (max_score - min_score)
        for layer, score in sensitivity_scores.items()
    }

    # Assign bit precision based on thresholds
    precision_mapping = {}
    for layer, norm_score in normalized_scores.items():
        if norm_score >= thresholds[0]:  # High sensitivity
            precision_mapping[layer] = bit_precisions[0]
        elif norm_score >= thresholds[1]:  # Medium sensitivity
            precision_mapping[layer] = bit_precisions[1]
        else:  # Low sensitivity
            precision_mapping[layer] = bit_precisions[2]

    return precision_mapping

# Example Usage
bit_precision_mapping = assign_bit_precision(
    sensitivity_scores=sensitivities,
    bit_precisions=(8, 4, 2),  # Define precision levels
    thresholds=(0.8, 0.2)  # Top 20% for 8-bit, bottom 20% for 2-bit
)

# Print layer-wise bit precision
print("Layer-wise Bit Precision:")
for layer, precision in bit_precision_mapping.items():
    print(f"{layer}: {precision}-bit")


In [ ]:
import matplotlib.pyplot as plt
scores = list(sensitivities.values())
plt.hist(scores, bins=20)
plt.xlabel("Sensitivity Score")
plt.ylabel("Frequency")
plt.title("Distribution of Sensitivity Scores")
plt.show()


In [ ]:
!/usr/src/tensorrt/bin/trtexec

In [ ]:
# Exporting to ONNX
dummy_input = torch.randn(1, 3, 512, 1024, device='cuda')
input_names = ["actual_input_1"]
output_names = ["output1"]
torch.onnx.export(
    model,
    dummy_input,
    "/kaggle/working/deeplab_resnet50_base.onnx",  # Single .onnx extension
    verbose=False,
    opset_version=11,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names
)

# Converting ONNX model to TRT
! /usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/deeplab_resnet50_base.onnx --useSpinWait --avgRuns=100 --fp32  --saveEngine=/kaggle/working/deeplab_resnet50_base.trt 

In [ ]:
sensitivities

In [ ]:
import torch
import numpy as np

# Define normalization parameters
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()  # For PyTorch
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

# Generate dynamic input tensor and normalize
dummy_input = torch.randn(1, 3, 512, 1024, device='cuda')  # Dynamic size
normalized_input = (dummy_input - mean) / std

# Specify input and output names
input_names = ["actual_input_1"]
output_names = ["output1"]


# Export the model using the normalized input
torch.onnx.export(
    model.eval(),
    normalized_input,  # Pass the normalized input for export
    "/kaggle/working/mobilenetv3_base_bs1.onnx",
    verbose=False,
    opset_version=11,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names,
    # dynamic_axes=dynamic_axes
)

# Convert ONNX model to TensorRT with dynamic shapes
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv3_base_bs1.onnx  --useSpinWait --workspace=512 --avgRuns=100 --saveEngine=/kaggle/working/mobilenetv3_base_bs1.trt


In [ ]:
# Define normalization parameters
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()  # For PyTorch
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

# Generate dynamic input tensor and normalize
dummy_input = torch.randn(1, 3, 512, 1024, device='cuda')  # Dynamic size
normalized_input = (dummy_input - mean) / std
# Specify input and output names
input_names = ["actual_input_1"]
output_names = ["output1"]

dynamic_axes = {
    "actual_input_1": {2: "height", 3: "width"},  # Dynamic height and width
    "output1": {2: "height", 3: "width"}
}

torch.onnx.export(
    model.eval(),
    dummy_input,  # Replace with a valid input tensor
    "/kaggle/working/mobilenetv3_base_dynamic.onnx",
    verbose=False,
    opset_version=11,
    do_constant_folding=True,
    input_names=["actual_input_1"],
    output_names=["output1"],
    dynamic_axes=dynamic_axes
)

!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv3_base_dynamic.onnx --minShapes=actual_input_1:1x3x256x512 --optShapes=actual_input_1:1x3x512x1024 --maxShapes=actual_input_1:1x3x1024x2048 --workspace=512 --avgRuns=100 --saveEngine=/kaggle/working/mobilenetv3_base_dynamic.trt 
 


In [ ]:
import torch

# Define normalization parameters
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()  # For PyTorch
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

# Generate dynamic input tensor and normalize
dummy_input = torch.randn(1, 3, 512, 1024, device='cuda')  # Dynamic size
normalized_input = (dummy_input - mean) / std
# Specify input and output names
input_names = ["actual_input_1"]
output_names = ["output1"]

# Export the model with dynamic input sizes
torch.onnx.export(
    model.eval(),
    dummy_input,
    "/kaggle/working/deeplab_resnet50_dynamic_half.onnx",  # Save as .onnx file
    verbose=False,
    opset_version=11,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names,
    # dynamic_axes=dynamic_axes  # Specify dynamic axes
)

# Convert ONNX model to TensorRT with dynamic shapes
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/deeplab_resnet50_dynamic_half.onnx  --fp16 --useSpinWait --workspace=512 --avgRuns=100 --saveEngine=/kaggle/working/deeplab_resnet50_dynamic_half.trt


In [ ]:
import torch

# Define dummy input with a fixed size for export, but specify dynamic dimensions
# Define normalization parameters
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()  # For PyTorch
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

# Generate dynamic input tensor and normalize
dummy_input = torch.randn(1, 3, 512, 1024, device='cuda')  # Dynamic size
normalized_input = (dummy_input - mean) / std

# Specify input and output names
input_names = ["actual_input_1"]
output_names = ["output1"]

# Export the model with dynamic input sizes
torch.onnx.export(
    model.eval(),
    dummy_input,
    "/kaggle/working/deeplab_resnet50_dynamic_int8.onnx",  # Save as .onnx file
    verbose=False,
    opset_version=11,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names,
    # dynamic_axes=dynamic_axes  # Specify dynamic axes
)

# Convert ONNX model to TensorRT with dynamic shapes
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/deeplab_resnet50_dynamic_int8.onnx --int8 --useSpinWait --workspace=512 --avgRuns=100 --saveEngine=/kaggle/working/deeplab_resnet50_dynamic_int8.trt


In [ ]:
#-----------per channel weight-----------------------------------
quant_desc_input = QuantDescriptor(num_bits=8, calib_method='histogram')
quant_nn.QuantConv2d.set_default_quant_desc_input(quant_desc_input)
quant_nn.QuantMaxPool2d.set_default_quant_desc_input(quant_desc_input)
quant_nn.QuantAdaptiveAvgPool2d.set_default_quant_desc_input(quant_desc_input)
quant_nn.QuantLinear.set_default_quant_desc_input(quant_desc_input)

# #-----------per tensor weight--------------------------------------
# quant_desc_input = QuantDescriptor(num_bits=4, calib_method='histogram', axis=None)
# quant_nn.QuantConv2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantMaxPool2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantConvTranspose2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantLinear.set_default_quant_desc_input(quant_desc_input)

# quant_desc_weight = QuantDescriptor(num_bits=4, calib_method='histogram', axis=None)
# quant_nn.QuantConv2d.set_default_quant_desc_weight(quant_desc_weight)
# quant_nn.QuantConvTranspose2d.set_default_quant_desc_weight(quant_desc_weight)
# quant_nn.QuantLinear.set_default_quant_desc_weight(quant_desc_weight)


In [ ]:
quant_modules.initialize()

In [ ]:
# Load the DeepLabV3-ResNet50 model pre-trained on COCO
# if accelerator.is_main_process:
q_model = models.segmentation.deeplabv3_resnet50(pretrained=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Modify the classifier to match the number of classes in Cityscapes (19 classes + 1 ignore class)
q_model.classifier[4] = torch.nn.Conv2d(256, 20, kernel_size=(1, 1), stride=(1, 1))
q_model.aux_classifier[4] = torch.nn.Conv2d(256, 20, kernel_size=(1, 1), stride=(1, 1))

# Wrap the model in DataParallel and move to main device
# if torch.cuda.device_count() > 1:
#     print("Using", torch.cuda.device_count(), "GPUs!")
#     q_model = nn.DataParallel(q_model)  # Wrap model in DataParallel

# Move the model to the GPU if available
# model = model.to(accelerator.device)
q_model = q_model.to(device)


In [ ]:
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    q_model = nn.DataParallel(q_model)  # Wrap model in DataParallel


In [ ]:
q_model

In [ ]:
import torch
from collections import OrderedDict  # Import OrderedDict


# Load the full checkpoint dictionary
checkpoint = torch.load("/kaggle/input/deeplabv3-resnet50-final/pytorch/default/1/deeplabv3_resnet50_baseline.pth")

# Remove the "module." prefix from the keys in state_dict if necessary
state_dict = checkpoint['model_state_dict']
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k.replace("module.", "")  # Remove `module.` prefix
    new_state_dict[name] = v

# Load the model state
q_model.load_state_dict(new_state_dict)

# Load the optimizer state (optional, only if you want to continue training)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Retrieve additional information
epoch = checkpoint['epoch']
val_accuracy = checkpoint['val_accuracy']

print(f"Model loaded from epoch {epoch} with validation accuracy {val_accuracy* 100:.2f}%")

# Move model to the appropriate device if needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_model = q_model.to(accelerator.device)

# Set model to evaluation mode if you're using it for inference
q_model.eval()


In [ ]:
def compute_amax(model, **kwargs):
    # Load calib result
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                if isinstance(module._calibrator, calib.MaxCalibrator):
                    module.load_calib_amax()
                else:
                    module.load_calib_amax(**kwargs)
            print(F"{name:40}: {module}")
    model.cuda()

def collect_stats(model, data_loader, num_batches):
    """Feed data to the network and collect statistics"""
    # Enable calibrators
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                module.disable_quant()
                module.enable_calib()
            else:
                module.disable()

    # Feed data to the network for collecting stats
    for i, (image, _) in tqdm(enumerate(data_loader), total=num_batches):
        model(image.cuda())
        if i >= num_batches:
            break

    # Disable calibrators
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                module.enable_quant()
                module.disable_calib()
            else:
                module.enable()

def calibrate_model(model, model_name, data_loader, num_calib_batch, calibrator, hist_percentile, out_dir):
    """
        Feed data to the network and calibrate.
        Arguments:
            model: classification model
            model_name: name to use when creating state files
            data_loader: calibration data set
            num_calib_batch: amount of calibration passes to perform
            calibrator: type of calibration to use (max/histogram)
            hist_percentile: percentiles to be used for historgram calibration
            out_dir: dir to save state files in
    """

    if num_calib_batch > 0:
        print("Calibrating model")
        with torch.no_grad():
            collect_stats(model, data_loader, num_calib_batch)

        if not calibrator == "histogram":
            compute_amax(model, method="max")
            calib_output = os.path.join(
                out_dir,
                F"{model_name}-max-{num_calib_batch*data_loader.batch_size}.pth")
            torch.save(model.state_dict(), calib_output)
        else:
            for percentile in hist_percentile:
                print(F"{percentile} percentile calibration")
                compute_amax(model, method="percentile")
                calib_output = os.path.join(
                    out_dir,
                    F"{model_name}-percentile-{percentile}-{num_calib_batch*data_loader.batch_size}.pth")
                torch.save(model.state_dict(), calib_output)

            for method in ["mse", "entropy"]:
                print(F"{method} calibration")
                compute_amax(model, method=method)
                calib_output = os.path.join(
                    out_dir,
                    F"{model_name}-{method}-{num_calib_batch*data_loader.batch_size}.pth")
                torch.save(model.state_dict(), calib_output)

In [ ]:
# Set model to evaluation mode if you're using it for inference
q_model.eval()

In [ ]:
!mkdir calib_cache

In [ ]:
#Calibrate the model using max calibration technique.
with torch.no_grad():
    calibrate_model(
        model=q_model,
        model_name="deeplabv3_resnet50_qat",
        data_loader=train_loader,
        num_calib_batch=32,
        calibrator="histogram",
        hist_percentile=[99.9], 
        out_dir="/kaggle/working/calib_cache/")
"""[ 99.99, 99.999, 99.9999]"""

In [ ]:
# Set static member of TensorQuantizer to use Pytorch’s own fake quantization functions
quant_nn.TensorQuantizer.use_fb_fake_quant = True

# Exporting to ONNX
dummy_input = torch.randn(1, 3, 512, 1024, device='cuda')
input_names = ["actual_input_1"]
output_names = ["output1"]
torch.onnx.export(
    model,
    dummy_input,
    "/kaggle/working/deeplab_resnet50_ptq.onnx",  # Single .onnx extension
    verbose=False,
    opset_version=11,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names
)

# Converting ONNX model to TRT
! /usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/deeplab_resnet50_ptq.onnx --useSpinWait --avgRuns=100  --saveEngine=/kaggle/working/deeplab_resnet50_ptq.trt

In [ ]:
import os

n_classes = 20
num_epochs = 5
patience = 5  # Stop training if no improvement for 5 epochs
early_stopping = EarlyStopping(patience=patience, min_delta=0.01)
best_val_accuracy = 0
log_path = '/kaggle/working/training_log.txt'

if accelerator.is_main_process:
    # Open log file only on the main process
    with open(log_path, 'w') as log_file:
        log_file.write("Epoch, Train Loss, Train Acc, Val Loss, Val Acc, Val IoU\n")

# Training loop with early stopping
for epoch in range(num_epochs):
    # Train the model for one epoch
    train_loss, train_accuracy = train_one_epoch(q_model, train_loader, criterion, optimizer, accelerator.device)
    val_loss, val_accuracy, val_iou = evaluate_model_with_metrics(q_model, val_loader, criterion, accelerator.device, n_classes)

    # Print metrics for the current epoch (on main process only)
    if accelerator.is_main_process:
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}")
        print(f"Validation Loss: {val_loss:.4f}, Validation Acc: {val_accuracy:.4f}, Validation IoU: {val_iou:.4f}")

        # Log metrics to file
        with open(log_path, 'a') as log_file:
            log_file.write(f"{epoch + 1}, {train_loss:.4f}, {train_accuracy:.4f}, {val_loss:.4f}, {val_accuracy:.4f}, {val_iou:.4f}\n")

        # Save the model checkpoint if validation accuracy improves
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            save_checkpoint(q_model, optimizer, epoch, val_accuracy, path='/kaggle/working/deeplabv3_resnet50_qat.pth', accelerator=accelerator)

    # Early stopping
    early_stopping(val_accuracy)
    if early_stopping.early_stop:
        if accelerator.is_main_process:
            print("Early stopping triggered. Stopping training.")
        break

    # Step the scheduler if using a plateau scheduler
    # scheduler.step(val_loss)

if accelerator.is_main_process:
    print("Training complete!")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_quantization import nn as quant_nn
from pytorch_quantization.tensor_quant import QuantDescriptor

class QuantConv2d(nn.Conv2d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True):
        super(QuantConv2d, self).__init__(in_channels, out_channels, kernel_size, stride, padding, dilation, groups, bias)
        # Initialize input and weight quantizers with default 8-bit precision
        self.input_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='histogram', learn_amax=True))
        self.weight_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='histogram', learn_amax=True))

    def forward(self, input):
        # Quantize the input and weights before applying convolution
        quant_input = self.input_quantizer(input)
        quant_weight = self.weight_quantizer(self.weight)
        return F.conv2d(quant_input, quant_weight, self.bias, self.stride, self.padding, self.dilation, self.groups)

    def update_num_bits(self, num_bits_input, num_bits_weight):
        # Update the bit-width for input and weight quantizers
        self.input_quantizer.num_bits = num_bits_input
        self.weight_quantizer.num_bits = num_bits_weight


class QuantLinear(nn.Linear):
    def __init__(self, in_features, out_features, bias=True):
        super(QuantLinear, self).__init__(in_features, out_features, bias)
        # Initialize input and weight quantizers with default 8-bit precision
        self.input_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='histogram', learn_amax=True))
        self.weight_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='histogram', learn_amax=True))

    def forward(self, input):
        # Quantize the input and weights before applying the linear transformation
        quant_input = self.input_quantizer(input)
        quant_weight = self.weight_quantizer(self.weight)
        return F.linear(quant_input, quant_weight, self.bias)

    def update_num_bits(self, num_bits_input, num_bits_weight):
        # Update the bit-width for input and weight quantizers
        self.input_quantizer.num_bits = num_bits_input
        self.weight_quantizer.num_bits = num_bits_weight


In [ ]:
def update_quantization_bits_with_configs(model, configs, parent_name=""):
    """
    Update the quantization bit-widths of specific layers based on a configuration dictionary.
    
    Args:
        model (nn.Module): The model to update.
        configs (dict): Dictionary where keys are layer names and values are desired bit-widths.
                        Example: {'conv1': 8, 'layer1.0.conv1': 8, 'layer1.0.conv2': 4}
        parent_name (str): The name of the parent module (for recursive tracking).
    
    Returns:
        nn.Module: The updated model with the appropriate bit-widths applied.
    """
    for name, module in model.named_children():
        # Construct the full layer name for nested layers
        layer_name = f"{parent_name}.{name}" if parent_name else name

        # Check if the layer_name exists in the configs dict
        if layer_name in configs:
            num_bits = configs[layer_name]

            # Update bits if the module has quantizers
            if hasattr(module, 'input_quantizer') and isinstance(module.input_quantizer, TensorQuantizer):
                module.input_quantizer.num_bits = 8
                print(f"Updated input quantizer for {layer_name} to {num_bits}-bit.")
            
            if hasattr(module, 'weight_quantizer') and isinstance(module.weight_quantizer, TensorQuantizer):
                module.weight_quantizer.num_bits = num_bits
                print(f"Updated weight quantizer for {layer_name} to {num_bits}-bit.")

        # Recurse into child modules if they exist
        if list(module.children()):  # Check if module has children
            update_quantization_bits_with_configs(module, configs, layer_name)
    
    return model


In [ ]:
q_model = update_quantization_bits_with_configs(q_model, configs, parent_name="")

In [ ]:
configs = {
    # Initial Conv layer and Stem
    'backbone.conv1': 8,
    'backbone.bn1': 8,
    'backbone.layer1.0.conv1': 8,
    'backbone.layer1.0.bn1': 8,
    'backbone.layer1.0.conv2': 8,
    'backbone.layer1.0.bn2': 8,
    'backbone.layer1.0.conv3': 8,
    'backbone.layer1.0.bn3': 8,
    
    # Layer 1
    'backbone.layer1.1.conv1': 8,
    'backbone.layer1.1.bn1': 8,
    'backbone.layer1.1.conv2': 8,
    'backbone.layer1.1.bn2': 8,
    'backbone.layer1.1.conv3': 8,
    'backbone.layer1.1.bn3': 8,
    'backbone.layer1.2.conv1': 8,
    'backbone.layer1.2.bn1': 8,
    'backbone.layer1.2.conv2': 8,
    'backbone.layer1.2.bn2': 8,
    'backbone.layer1.2.conv3': 8,
    'backbone.layer1.2.bn3': 8,

    # Layer 2
    'backbone.layer2.0.conv1': 4,
    'backbone.layer2.0.bn1': 4,
    'backbone.layer2.0.conv2': 4,
    'backbone.layer2.0.bn2': 4,
    'backbone.layer2.0.conv3': 4,
    'backbone.layer2.0.bn3': 4,
    'backbone.layer2.1.conv1': 4,
    'backbone.layer2.1.bn1': 4,
    'backbone.layer2.1.conv2': 4,
    'backbone.layer2.1.bn2': 4,
    'backbone.layer2.1.conv3': 4,
    'backbone.layer2.1.bn3': 4,
    
    # Layer 3 (with dilated convolutions in DeepLabV3)
    'backbone.layer3.0.conv1': 4,
    'backbone.layer3.0.bn1': 4,
    'backbone.layer3.0.conv2': 4,
    'backbone.layer3.0.bn2': 4,
    'backbone.layer3.0.conv3': 4,
    'backbone.layer3.0.bn3': 4,
    'backbone.layer3.1.conv1': 4,
    'backbone.layer3.1.bn1': 4,
    'backbone.layer3.1.conv2': 4,
    'backbone.layer3.1.bn2': 4,
    'backbone.layer3.1.conv3': 4,
    'backbone.layer3.1.bn3': 4,
    
    # Layer 4 (higher dilation rate for larger context in DeepLabV3)
    'backbone.layer4.0.conv1': 8,
    'backbone.layer4.0.bn1': 8,
    'backbone.layer4.0.conv2': 8,
    'backbone.layer4.0.bn2': 8,
    'backbone.layer4.0.conv3': 8,
    'backbone.layer4.0.bn3': 8,
    'backbone.layer4.1.conv1': 8,
    'backbone.layer4.1.bn1': 8,
    'backbone.layer4.1.conv2': 8,
    'backbone.layer4.1.bn2': 8,
    'backbone.layer4.1.conv3': 8,
    'backbone.layer4.1.bn3': 8,
    
    # ASPP (Atrous Spatial Pyramid Pooling) Module
    'classifier.aspp.conv_1x1_1': 8,
    'classifier.aspp.conv_3x3_1': 4,
    'classifier.aspp.conv_3x3_2': 4,
    'classifier.aspp.conv_3x3_3': 4,
    'classifier.aspp.image_pool': 8,
    'classifier.aspp.project': 8,
    
    # Final Classifier layers
    'classifier.0': 8,  # Classifier conv layer
    'classifier.1': 8,  # Output layer (e.g., final fully connected)
}


In [ ]:
import time
import torch

def calculate_throughput(model, input_shape=(8, 3, 512, 1024), nwarmup=50, nruns=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    input_data = torch.randn(input_shape).to(device)
    
    # Warm-up
    with torch.no_grad():
        for _ in range(nwarmup):
            _ = model(input_data)
    torch.cuda.synchronize()
    
    # Measure throughput
    timings = []
    with torch.no_grad():
        for _ in range(nruns):
            start_time = time.time()
            _ = model(input_data)
            torch.cuda.synchronize()
            end_time = time.time()
            timings.append(end_time - start_time)
    
    avg_batch_time = sum(timings) / nruns
    batch_size = input_shape[0]
    throughput = batch_size / avg_batch_time
    print(f"Average Throughput: {throughput:.2f} samples/second")
    return throughput


def calculate_latency(model, input_shape=(1, 3, 512, 1024), nwarmup=50, nruns=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    input_data = torch.randn(input_shape).to(device)
    
    # Warm-up
    with torch.no_grad():
        for _ in range(nwarmup):
            _ = model(input_data)
    torch.cuda.synchronize()
    
    # Measure latency
    timings = []
    with torch.no_grad():
        for _ in range(nruns):
            start_time = time.time()
            _ = model(input_data)
            torch.cuda.synchronize()  # Ensure all ops are completed
            end_time = time.time()
            timings.append(end_time - start_time)
    
    avg_latency = sum(timings) / nruns
    print(f"Average Latency: {avg_latency * 1000:.2f} ms")
    return avg_latency


def calculate_model_size(model, dtype='fp32'):
    num_params = sum(p.numel() for p in model.parameters())
    
    if dtype == 'fp32':
        param_size = 4  # FP32 = 4 bytes
    elif dtype == 'fp16':
        param_size = 2  # FP16 = 2 bytes
    elif dtype == 'int8':
        param_size = 1  # INT8 = 1 byte
    elif dtype == 'int4':
        param_size = 0.5  # INT4 = 0.5 bytes
    else:
        raise ValueError("Unsupported data type")
    
    model_size = num_params * param_size / (1024 ** 2)  # Convert to MB
    print(f"Model Size: {model_size:.2f} MB")
    return model_size


In [ ]:
throughput = calculate_throughput(model, input_shape=(1, 3, 512, 1024), nwarmup=50, nruns=100)
latency = calculate_latency(model, input_shape=(1, 3, 512, 1024), nwarmup=50, nruns=100)
model_size = calculate_model_size(model, dtype='fp16')  # Example for FP16


In [ ]:
def allocate_buffers(engine, max_batch_size, height, width, n_classes):
    inputs, outputs, bindings = [], [], []
    stream = cuda.Stream()
    for binding in range(engine.num_bindings):
        binding_shape = (
            (max_batch_size, 3, height, width) 
            if engine.binding_is_input(binding) 
            else (max_batch_size, n_classes, height, width)
        )
        size = trt.volume(binding_shape)
        dtype = trt.nptype(engine.get_binding_dtype(binding))
        host_mem = cuda.pagelocked_empty(size, dtype)
        device_mem = cuda.mem_alloc(host_mem.nbytes)
        bindings.append(int(device_mem))
        if engine.binding_is_input(binding):
            inputs.append({'host': host_mem, 'device': device_mem})
        else:
            outputs.append({'host': host_mem, 'device': device_mem})
    return inputs, outputs, bindings, stream

def infer_and_evaluate(engine, context, val_loader, criterion, device, n_classes, max_batch_size, height, width):
    # Allocate buffers once outside the loop
    inputs, outputs, bindings, stream = allocate_buffers(engine, max_batch_size, height, width, n_classes)
    total_loss, total_accuracy, total_iou, latencies = 0.0, 0.0, 0.0, []
    total_images = 0

    with torch.no_grad():
        for images, masks in val_loader:
            batch_size, channels, h, w = images.shape
            images, masks = images.to(device), masks.to(device).long()

            # Set dynamic shape for context
            context.set_binding_shape(0, (batch_size, channels, h, w))

            # Copy images to input buffer
            images = images.cpu().numpy().astype(np.float32)
            np.copyto(inputs[0]['host'], images.ravel())

            # Measure latency
            start_time = time.time()
            cuda.memcpy_htod_async(inputs[0]['device'], inputs[0]['host'], stream)
            context.execute_async_v2(bindings=bindings, stream_handle=stream.handle)
            cuda.memcpy_dtoh_async(outputs[0]['host'], outputs[0]['device'], stream)
            stream.synchronize()
            latency = time.time() - start_time
            latencies.append(latency)
            total_images += batch_size

            # Process model output
            output_shape = (batch_size, n_classes, h, w)  # Adjust based on context output
            main_output = torch.from_numpy(np.array(outputs[0]['host'])).view(*output_shape).to(device)
            loss = criterion(main_output, masks)
            accuracy = calculate_accuracy(main_output, masks)
            _, preds = torch.max(main_output, 1)
            iou = calculate_mean_iou(preds, masks, n_classes)

            total_loss += loss.item()
            total_accuracy += accuracy.item()
            total_iou += iou

    # Calculate average metrics
    avg_loss = total_loss / len(val_loader)
    avg_accuracy = total_accuracy / len(val_loader)
    avg_iou = total_iou / len(val_loader)
    avg_latency = np.mean(latencies)
    throughput = total_images / sum(latencies)

    return avg_loss, avg_accuracy, avg_iou, avg_latency, throughput


In [ ]:
from PIL import Image
import numpy as np
import torch
from torch.utils.data import DataLoader
import time
import pycuda.driver as cuda
import tensorrt as trt

# Define mean and std for normalization
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def normalize_input(image):
    """
    Normalize the input image using mean and std.
    Input shape: (C, H, W)
    """
    image = (image - mean[:, None, None]) / std[:, None, None]
    return image

def allocate_buffers(engine, batch_size, height, width, n_classes):
    """
    Allocate buffers for TensorRT inference.
    """
    inputs, outputs, bindings = [], [], []
    stream = cuda.Stream()
    for binding in range(engine.num_bindings):
        binding_shape = (
            (batch_size, 3, height, width) 
            if engine.binding_is_input(binding) 
            else (batch_size, n_classes, height, width)
        )
        size = trt.volume(binding_shape)
        dtype = trt.nptype(engine.get_binding_dtype(binding))
        host_mem = cuda.pagelocked_empty(size, dtype)
        device_mem = cuda.mem_alloc(host_mem.nbytes)
        bindings.append(int(device_mem))
        if engine.binding_is_input(binding):
            inputs.append({'host': host_mem, 'device': device_mem})
        else:
            outputs.append({'host': host_mem, 'device': device_mem})
    return inputs, outputs, bindings, stream

def infer_and_evaluate(engine, context, val_loader, criterion, device, n_classes):
    """
    Run inference and evaluate the model using validation data.
    Assumes batch size = 1 and image shape = 512 x 1024.
    """
    # Fixed parameters for batch size and image dimensions
    batch_size = 1
    height, width = 512, 1024

    # Allocate buffers once
    inputs, outputs, bindings, stream = allocate_buffers(engine, batch_size, height, width, n_classes)

    total_loss, total_accuracy, total_iou, latencies = 0.0, 0.0, 0.0, []
    total_images = 0

    with torch.no_grad():
        for images, masks in val_loader:
            # Ensure batch size = 1
            images, masks = images[:batch_size].to(device), masks[:batch_size].to(device).long()

            # Normalize images
            images = images.cpu().numpy()  # Convert to NumPy for normalization
            images = np.array([normalize_input(img) for img in images])  # Normalize each image
            images = torch.from_numpy(images).to(device)  # Convert back to tensor

            # Copy images to input buffer
            np.copyto(inputs[0]['host'], images.cpu().numpy().ravel())

            # Measure latency
            start_time = time.time()
            cuda.memcpy_htod_async(inputs[0]['device'], inputs[0]['host'], stream)
            context.execute_async_v2(bindings=bindings, stream_handle=stream.handle)
            cuda.memcpy_dtoh_async(outputs[0]['host'], outputs[0]['device'], stream)
            stream.synchronize()
            latency = time.time() - start_time
            latencies.append(latency)
            total_images += batch_size

            # Process model output
            output_shape = (batch_size, n_classes, height, width)
            main_output = torch.from_numpy(np.array(outputs[0]['host'])).view(*output_shape).to(device)
            loss = criterion(main_output, masks)
            accuracy = calculate_accuracy(main_output, masks)
            _, preds = torch.max(main_output, 1)
            iou = calculate_mean_iou(preds, masks, n_classes)

            total_loss += loss.item()
            total_accuracy += accuracy.item()
            total_iou += iou

    # Calculate average metrics
    avg_loss = total_loss / len(val_loader)
    avg_accuracy = total_accuracy / len(val_loader)
    avg_iou = total_iou / len(val_loader)
    avg_latency = np.mean(latencies)
    throughput = total_images / sum(latencies)  # This will equal FPS for batch_size = 1

    return avg_loss, avg_accuracy, avg_iou, avg_latency, throughput


In [ ]:
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit  # Automatically initializes CUDA driver


# Define necessary parameters
engine_path = "/kaggle/working/deeplab_resnet50_dynamic_half.trt"
device = torch.device("cuda")  # or torch.device("cpu")
n_classes = 19  # Set according to your dataset, e.g., 19 for Cityscapes

# Load the TensorRT engine
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

def load_engine(engine_path):
    """
    Load a serialized TensorRT engine from file.
    """
    with open(engine_path, "rb") as f, trt.Runtime(TRT_LOGGER) as runtime:
        engine = runtime.deserialize_cuda_engine(f.read())
        if engine is None:
            raise RuntimeError(f"Failed to load TensorRT engine from {engine_path}")
        return engine

engine = load_engine(engine_path)

# Create an execution context from the engine
context = engine.create_execution_context()

# Ensure context supports dynamic input shapes (if applicable)
if not context:
    raise RuntimeError("Failed to create execution context from TensorRT engine.")

# Call the function
avg_loss, avg_accuracy, avg_iou, avg_latency, throughput = infer_and_evaluate(engine, context, inference_loader, criterion, device, n_classes)

# Print the results
print(f"Average Loss: {avg_loss:.4f}")
print(f"Average Accuracy: {avg_accuracy:.4f}")
print(f"Average IoU: {avg_iou:.4f}")
print(f"Average Latency: {avg_latency:.4f} seconds")
print(f"Throughput (FPS): {throughput:.2f} frames/sec")


In [ ]:
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit
import torch
import numpy as np
import time

# Define necessary parameters
ENGINE_PATH = "/kaggle/working/mobilenetv3_base_dynamic.trt"
DEVICE = torch.device("cuda")
N_CLASSES = 19  # Number of classes (e.g., 19 for Cityscapes)

# TensorRT Logger
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

# Function to load TensorRT engine
def load_engine(engine_path):
    with open(engine_path, "rb") as f, trt.Runtime(TRT_LOGGER) as runtime:
        engine = runtime.deserialize_cuda_engine(f.read())
        if engine is None:
            raise RuntimeError(f"Failed to load TensorRT engine from {engine_path}")
        return engine

# Function to allocate buffers
def allocate_buffers(engine):
    inputs, outputs, bindings = [], [], []
    stream = cuda.Stream()
    for binding in range(engine.num_bindings):
        dtype = trt.nptype(engine.get_binding_dtype(binding))
        bindings.append(None)  # Placeholder for device memory
        if engine.binding_is_input(binding):
            inputs.append({'host': None, 'device': None, 'dtype': dtype})
        else:
            outputs.append({'host': None, 'device': None, 'dtype': dtype})
    return inputs, outputs, bindings, stream

# Function to resize buffers based on input shape
def resize_buffers(inputs, outputs, bindings, engine, context, input_shape):
    batch_size, _, height, width = input_shape
    for binding in range(engine.num_bindings):
        if engine.binding_is_input(binding):
            size = trt.volume((batch_size, 3, height, width))
        else:
            size = trt.volume((batch_size, N_CLASSES, height, width))
        
        dtype = trt.nptype(engine.get_binding_dtype(binding))
        host_mem = cuda.pagelocked_empty(size, dtype)
        device_mem = cuda.mem_alloc(host_mem.nbytes)
        bindings[binding] = int(device_mem)
        
        if engine.binding_is_input(binding):
            inputs[0]['host'] = host_mem
            inputs[0]['device'] = device_mem
        else:
            outputs[0]['host'] = host_mem
            outputs[0]['device'] = device_mem

    # Set dynamic input shape in the execution context
    context.set_binding_shape(0, input_shape)
    if not context.all_binding_shapes_specified:
        raise RuntimeError(f"Failed to set dynamic input shape: {input_shape}")

# Function to normalize input
def normalize_input(image):
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    image = (image - mean[:, None, None]) / std[:, None, None]
    return image

# Function to run inference and validate results
def infer_and_evaluate(engine, context, val_loader, criterion, device):
    inputs, outputs, bindings, stream = allocate_buffers(engine)

    total_loss, total_accuracy, total_iou, latencies = 0.0, 0.0, 0.0, []
    total_images = 0

    with torch.no_grad():
        for images, masks in val_loader:
            # Get input dimensions and batch size
            batch_size, _, height, width = images.shape
            input_shape = (batch_size, 3, height, width)

            # Normalize images
            images = images.cpu().numpy()  # Convert to NumPy for normalization
            images = np.array([normalize_input(img) for img in images])  # Normalize each image

            # Resize buffers and set dynamic input shape
            resize_buffers(inputs, outputs, bindings, engine, context, input_shape)

            # Copy images to input buffer
            np.copyto(inputs[0]['host'], images.ravel())

            # Measure latency
            start_time = time.time()
            cuda.memcpy_htod_async(inputs[0]['device'], inputs[0]['host'], stream)
            context.execute_async_v2(bindings=bindings, stream_handle=stream.handle)
            cuda.memcpy_dtoh_async(outputs[0]['host'], outputs[0]['device'], stream)
            stream.synchronize()
            latency = time.time() - start_time
            latencies.append(latency)
            total_images += batch_size

            # Process model output
            output_shape = (batch_size, N_CLASSES, height, width)
            main_output = torch.from_numpy(np.array(outputs[0]['host'])).view(*output_shape).to(device)
            loss = criterion(main_output, masks.to(device).long())
            accuracy = calculate_accuracy(main_output, masks.to(device).long())
            _, preds = torch.max(main_output, 1)
            iou = calculate_mean_iou(preds, masks.to(device).long(), N_CLASSES)

            total_loss += loss.item()
            total_accuracy += accuracy.item()
            total_iou += iou

    # Calculate average metrics
    avg_loss = total_loss / len(val_loader)
    avg_accuracy = total_accuracy / len(val_loader)
    avg_iou = total_iou / len(val_loader)
    avg_latency = np.mean(latencies)
    throughput = total_images / sum(latencies)  # This will equal FPS for batch_size = 1

    return avg_loss, avg_accuracy, avg_iou, avg_latency, throughput


# Load the TensorRT engine
engine = load_engine(ENGINE_PATH)
context = engine.create_execution_context()

# Example resolutions to test
resolutions = [(1024, 2048)]

# Example loop to verify resolutions and infer
for resolution in resolutions:
    print(f"Testing Resolution: {resolution}")
    # dummy_loader = generate_dummy_loader(resolution)  # Replace with real DataLoader
    avg_loss, avg_accuracy, avg_iou, avg_latency, throughput = infer_and_evaluate(
        engine, context, inference_loader, criterion, DEVICE
    )
    print(f"Resolution {resolution}:")
    print(f"  Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}, IoU: {avg_iou:.4f}")
    print(f"  Latency: {avg_latency:.4f} sec, Throughput: {throughput:.2f} FPS")


In [ ]:
# Load TensorRT engine
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
engine_path = "/kaggle/working/mobilenetv3_base_dynamic.trt"
with open(engine_path, "rb") as f, trt.Runtime(TRT_LOGGER) as runtime:
    engine = runtime.deserialize_cuda_engine(f.read())
context = engine.create_execution_context()

# Define resolutions for benchmarking
resolutions = [(256, 512), (512, 1024), (1024, 2048)]

# Run benchmarks
for height, width in resolutions:
    latency, fps = benchmark(engine, context, batch_size=1, height=height, width=width)
    print(f"Resolution: {height}x{width}, Latency: {latency:.4f}s, FPS: {fps:.2f}")


In [ ]:
#Full precision
Average Loss: 0.2129
Average Accuracy: 0.8161
Average IoU: 0.3594
Average Latency: 0.0166 seconds
Throughput (FPS): 60.19 frames/sec

#Dynamic Full precision (FP32)
Average Loss: 0.2129
Average Accuracy: 0.8161
Average IoU: 0.3594
Average Latency: 0.0152 seconds
Throughput (FPS): 65.58 frames/sec

#half precision (FP16)
Average Loss: 2.9444
Average Accuracy: 0.3294
Average IoU: 0.0196
Average Latency: 0.0079 seconds
Throughput (FPS): 126.53 frames/sec

#INT8 
Average Loss: 2.9444
Average Accuracy: 0.3294
Average IoU: 0.0196
Average Latency: 0.0064 seconds
Throughput (FPS): 157.17 frames/sec

In [ ]:
#FP32 batch-size=4 model-size=181MB workspace=512, (with workspace=1024 throughput = 1.23 (lower))
Average Loss: 0.16406167125701904
Average Accuracy: 0.8286482696533203
Average Mean IoU: 0.502413667396998
Average Latency (seconds): 0.5608027191162109
Throughput (images/second): 7.132633034133185

#FP16 img= 512 x 1024 model-size=84.61MB
Average Loss: 0.16486183202266694
Average Accuracy: 0.8284959411621093
Average Mean IoU: 0.5022109066484687
Average Latency (seconds): 0.11237919616699218
Throughput (images/second): 35.59377657459053

#FP16 img= 1024x 2048 model-size=84.61MB
Average Loss: 0.1653000465631485
Average Accuracy: 0.8285167007446289
Average Mean IoU: 0.5015127582253185
Average Latency (seconds): 0.11328376007080078
Throughput (images/second): 35.30956244301968

#INT8 img= 512x1024 
Average Loss: 0.16965551269054413
Average Accuracy: 0.8273351058959961
Average Mean IoU: 0.49953373773605486
Average Latency (seconds): 0.06106091117858887
Throughput (images/second): 65.50835752026262

#INT8 img= 1024 x 2024
Average Loss: 0.16927998000383376
Average Accuracy: 0.827403091430664
Average Mean IoU: 0.4999177049195407
Average Latency (seconds): 0.061386817932128905
Throughput (images/second): 65.16056923527978